# Fine-tune YOLO26 trên dataset xe Việt Nam (Kaggle GPU)

Notebook chạy trên GPU của Kaggle, đẩy lên bằng `python scripts/kaggle_train.py push` (không dùng extension Kaggle Studio: lỗi trên Windows).
Dataset `vn-vehicles-yolo` (tạo bởi `scripts/prepare_data.py`, upload bởi `scripts/kaggle_push_dataset.py`) được gắn sẵn qua `kernel-metadata.json`.

**Cách dùng**
1. Lần đầu: đặt `SMOKE_TEST = True` → `kaggle_train.py push` (~5 phút) để chắc chắn mọi thứ chạy được.
2. Đặt `SMOKE_TEST = False` → `kaggle_train.py push` để train thật (YOLO26n: ước tính 1–2 giờ trên T4).
3. Tải kết quả về bằng `python scripts/kaggle_train.py download`.

**Kết quả trong `/kaggle/working/`**
- `runs/<RUN_NAME>/` (mỗi seed một RUN_NAME) — weights, biểu đồ loss/mAP, `results.csv` theo từng epoch
- `<RUN_NAME>_best.pt` — trọng số tốt nhất (theo mAP trên val)
- `<RUN_NAME>_test_metrics.csv` — precision/recall/mAP trên tập test, theo từng lớp (bảng E1 cho báo cáo)
- `<RUN_NAME>_run_config.json` — cấu hình + phiên bản thư viện + GPU đã dùng (để tái lập)

## 1. Cấu hình thí nghiệm — chỉ sửa ở ô này

In [ ]:
MODEL = "yolo26n.pt"      # yolo26n.pt: nhanh, dùng cho demo trên CPU | yolo26s.pt: chính xác hơn, chậm hơn
EPOCHS = 100
PATIENCE = 20             # dừng sớm nếu mAP trên val không cải thiện sau 20 epoch liên tiếp
IMGSZ = 960               # v1 640; v2/v3/v5 960. v4 thử 1280: mAP không tăng, chậm hơn 29% -> v5 giữ 960 như v3
                          # để chỉ khác v3 ở dữ liệu (thêm UVH-26). Suy luận local phải dùng cùng imgsz.
BATCH = 16                # bản n @960 batch 16 dùng 10.7GB trên T4 15GB; bản s phải giảm còn 8 (như v4 @1280);
                          # ultralytics cộng dồn gradient tới batch hiệu dụng nbs=64 nên batch 8 hay 16 tương đương
SEEDS = [1, 2]            # v1–v5 và v5s đều train một lần với seed 42. Lặp lại v5 với seed khác để đo độ dao động
                          # giữa các lần train (mỗi seed ~3 giờ T4, chạy lần lượt trong cùng một phiên)
SMOKE_TEST = False        # True: chạy thử 3 epoch trên 10% dữ liệu để kiểm tra pipeline
ULTRALYTICS_VERSION = "8.4.163"  # cùng phiên bản với máy local để weights dùng lại được

RUN_TAG = "v5"            # phiên bản dataset/thí nghiệm — đổi mỗi lần để kết quả các lần chạy không ghi đè nhau
DEFAULT_SEED = 42         # seed của các lần train trước: giữ tên cũ (không hậu tố) cho seed này


def run_name(seed: int) -> str:
    """Tên thư mục/tệp kết quả của một lần train; seed khác 42 thêm hậu tố _seed<n>."""
    name = f"{MODEL.removesuffix('.pt')}_{RUN_TAG}_e{EPOCHS}_{IMGSZ}"
    if seed != DEFAULT_SEED:
        name += f"_seed{seed}"
    return name + ("_smoke" if SMOKE_TEST else "")


print("RUN_NAME =", [run_name(seed) for seed in SEEDS])

## 2. Cài thư viện và kiểm tra GPU

Kaggle có thể cấp GPU T4 hoặc P100. PyTorch bản mới đã bỏ hỗ trợ GPU đời cũ (P100 = sm_60), nên kiểm tra **trong tiến trình con**
trước khi `import torch`: nếu không tương thích thì cài PyTorch 2.5.1 (CUDA 12.1) — bản cuối còn hỗ trợ sm_60.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS_VERSION}"], check=True)

CHECK_GPU = """
import sys, torch
if not torch.cuda.is_available():
    sys.exit(2)
major, minor = torch.cuda.get_device_capability(0)
sys.exit(0 if f"sm_{major}{minor}" in torch.cuda.get_arch_list() else 1)
"""
status = subprocess.run([sys.executable, "-c", CHECK_GPU]).returncode
if status == 2:
    raise RuntimeError("Không có GPU: vào Settings của notebook trên Kaggle, chọn Accelerator = GPU.")
if status == 1:
    print("PyTorch hiện tại không hỗ trợ GPU này -> cài PyTorch 2.5.1 (CUDA 12.1)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.5.1", "torchvision==0.20.1",
                    "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)

import torch
import ultralytics

GPU_NAME = torch.cuda.get_device_name(0)
print(f"GPU: {GPU_NAME} x{torch.cuda.device_count()} | torch {torch.__version__} | ultralytics {ultralytics.__version__}")

## 3. Tìm dataset đã gắn vào notebook

In [ ]:
import shutil
from pathlib import Path

data_yamls = sorted(Path("/kaggle/input").glob("**/data.yaml"))
assert data_yamls, "Không thấy data.yaml trong /kaggle/input: kiểm tra dataset vn-vehicles-yolo đã được gắn (Add Input)."

# /kaggle/input chỉ đọc và đọc chậm (~4 MB/s, ultralytics không ghi được cache nhãn) ->
# chép dataset (~230 MB, vài giây) sang ổ local /kaggle/working rồi train từ đó.
LOCAL_DATASET = Path("/kaggle/working/vn_vehicles")
if not LOCAL_DATASET.exists():
    shutil.copytree(data_yamls[0].parent, LOCAL_DATASET)
DATA_YAML = LOCAL_DATASET / "data.yaml"
print("Dataset:", data_yamls[0].parent, "->", LOCAL_DATASET)
print((LOCAL_DATASET / "stats.csv").read_text())

## 4. Huấn luyện

Bắt đầu từ trọng số pretrained COCO (transfer learning). `data.yaml` không có khoá `path`, nên ultralytics lấy thư mục chứa nó làm gốc.

In [ ]:
from ultralytics import YOLO

OUTPUT_DIR = Path("/kaggle/working")


def train(seed: int) -> Path:
    """Train một lần với seed cho trước, trả về thư mục kết quả runs/<RUN_NAME>."""
    model = YOLO(MODEL)
    model.train(
        data=str(DATA_YAML),
        epochs=3 if SMOKE_TEST else EPOCHS,
        fraction=0.1 if SMOKE_TEST else 1.0,
        patience=PATIENCE,
        imgsz=IMGSZ,
        batch=BATCH,
        seed=seed,
        deterministic=True,
        device=0,
        workers=4,
        project=str(OUTPUT_DIR / "runs"),
        name=run_name(seed),
        exist_ok=True,
    )
    return OUTPUT_DIR / "runs" / run_name(seed)

## 5. Đánh giá trên tập test

Tập test chỉ gồm các cảnh **không xuất hiện trong train** (xem `datasets_config.yaml` trong dataset), nên đây là con số tổng quát hoá thật.

In [ ]:
import csv

METRIC_NAMES = ["precision", "recall", "mAP50", "mAP50-95"]


def evaluate_test(run_dir: Path, name: str) -> None:
    """Chấm best.pt trên tập test, ghi <RUN_NAME>_test_metrics.csv theo từng lớp."""
    best_model = YOLO(run_dir / "weights" / "best.pt")
    metrics = best_model.val(
        data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH, device=0,
        project=str(run_dir), name="test", exist_ok=True,
    )
    rows = [{"class": "all", **dict(zip(METRIC_NAMES, metrics.box.mean_results()))}]
    for i, class_id in enumerate(metrics.ap_class_index):
        rows.append({"class": best_model.names[int(class_id)], **dict(zip(METRIC_NAMES, metrics.box.class_result(i)))})

    metrics_path = OUTPUT_DIR / f"{name}_test_metrics.csv"
    with metrics_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["class", *METRIC_NAMES])
        writer.writeheader()
        writer.writerows({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()} for row in rows)
    print(metrics_path.read_text())

## 6. Lưu trọng số và cấu hình đã dùng, rồi chạy lần lượt từng seed

In [ ]:
import json
import shutil


def save_outputs(run_dir: Path, name: str, seed: int) -> None:
    """Chép best.pt ra ngoài và ghi cấu hình đã dùng (để tái lập)."""
    shutil.copy2(run_dir / "weights" / "best.pt", OUTPUT_DIR / f"{name}_best.pt")
    run_config = {
        "run_name": name, "model": MODEL, "epochs": EPOCHS, "patience": PATIENCE, "imgsz": IMGSZ,
        "batch": BATCH, "seed": seed, "smoke_test": SMOKE_TEST, "dataset": str(DATA_YAML.parent),
        "gpu": GPU_NAME, "torch": torch.__version__, "ultralytics": ultralytics.__version__,
    }
    (OUTPUT_DIR / f"{name}_run_config.json").write_text(json.dumps(run_config, indent=2, ensure_ascii=False))


for seed in SEEDS:
    name = run_name(seed)
    run_dir = train(seed)
    evaluate_test(run_dir, name)
    save_outputs(run_dir, name, seed)

for path in sorted(OUTPUT_DIR.glob("*_run_config.json")):
    print(path.name)
shutil.rmtree(LOCAL_DATASET, ignore_errors=True)  # không đưa bản sao dataset vào output